In [46]:
import pandas as pd
import json
from openai import OpenAI
from tqdm import tqdm

In [47]:
# Asetukset
INPUT_CSV = "/xxx/ep_enlargement.csv"
OUTPUT_CSV = "llm_results_new_dataset_2105.csv"
MODEL = "gpt-5.4-mini"
N_TEST = len(df)
N_JUSTIFICATIONS = 0

client = OpenAI(api_key="xxx")

In [48]:
df = pd.read_csv(INPUT_CSV, sep=",", encoding="utf-8")

# Poista whitespace sarakenimistä
df.columns = df.columns.str.strip()


# Pvm fix
df["Date"] = pd.to_datetime(
    df["Date"],
    format="%Y%m%d",
    errors="coerce"
).dt.strftime("%Y-%m-%d")

# df["Date"] = pd.to_datetime(df["Date"], errors="coerce").dt.strftime("%Y-%m-%d")

# Tarkista
print(df.columns)
print(df.head(2))
print(df["Quote"].iloc[0][:200])



Index(['Date', 'Debate ID', 'Name', 'Type', 'Party', 'Origin', 'Country',
       'Position', 'Quote', 'postinvasion', 'Pos.Reg', 'Region'],
      dtype='object')
         Date     Debate ID                Name                         Type  \
0  2019-02-12  20190212-016  Cristian Dan Preda         Member of Parliament   
1  2019-02-12  20190212-016       George Ciamba  Member of Council of the EU   

                                        Party   Origin             Country  \
0  Group of the European People's Party (EPP)  Romania  Bosnia-Herzegovina   
1                       Other - Not specified  Romania  Bosnia-Herzegovina   

   Position                                              Quote  postinvasion  \
0         0  All of this is, in fact, within reach, provide...             0   
1         0  Let me start by reiterating our strong commitm...             0   

   Pos.Reg   Region  
0        2  Balkans  
1        2  Balkans  
All of this is, in fact, within reach, provided that th

In [1]:
#print(df.columns)
#print(df.head(3))

In [50]:
print(df.iloc[0])

Date                                                   2019-02-12
Debate ID                                            20190212-016
Name                                           Cristian Dan Preda
Type                                         Member of Parliament
Party                  Group of the European People's Party (EPP)
Origin                                                    Romania
Country                                        Bosnia-Herzegovina
Position                                                        0
Quote           All of this is, in fact, within reach, provide...
postinvasion                                                    0
Pos.Reg                                                         2
Region                                                    Balkans
Name: 0, dtype: object


In [51]:
# Prompt
SYSTEM_PROMPT = """You are a political science research assistant.
Identify which analytical frames related to the European union enlargement are present in the speech.
Be conservative: only mark a frame as present if it is clearly expressed. Avoid overclassification.
"""

USER_PROMPT_TEMPLATE = """
Speech:
\"\"\"
{speech}
\"\"\"

Determine whether EACH frame is present (true/false). There can be multiple frames.
If rule of law is mentioned as an accession criterion or benchmark, prefer procedural. If it is invoked as a European value or identity marker, prefer normative.

Frames:
1. geopolitical
Focus on common security and defence, geopolitics, war, instability, strategic stability
Typical signals: references to Russia or China, conflict, security threats, geopolical balance, defence policy.

2. economic
Focus on single market, cost of the enlargement, funding, subsidies, investment opportunities
Typical signals: economic costs and benefits associated with enlargement.

3. normative
Focus on values such as democracy, rule of law, human rights
Typical signals: talk about democracy, European values, European family, rule of law.

4. institutional
Focus on internal issues and reforms, EU institutions, efficiency, internal consolidation, deepening vs. widening
Typical signals: EU's internal functioning and absorption capacity.

5. procedural
Focus on accession steps and progress, benchmarks, alignment with EU acquis, merit-based accession, conditionality
Typical signals: Enlargement takes place within a detailed set of procedures, process is closely monitored.

6. If none apply, set all to false and "none" to true.


Respond STRICTLY in JSON:
{{
  "geopolitical": true/false,
  "economic": true/false,
  "normative": true/false,
  "institutional": true/false,
  "procedural": true/false,
  "none": true/false
}}
"""


In [52]:
results = []

for idx, row in tqdm(df.head(N_TEST).iterrows(), total=min(len(df), N_TEST)):

    speech = row["Quote"]

    if pd.isna(speech) or not str(speech).strip():
        continue

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": USER_PROMPT_TEMPLATE.format(speech=speech)}
        ],
        temperature=0
    )

    content = response.choices[0].message.content

    try:
        parsed = json.loads(content)
    except:
        parsed = {
            "geopolitical": False,
            "economic": False,
            "normative": False,
            "institutional": False,
            "procedural": False,
            "none": True
        }


    results.append({

        "Date": row.get("Date"),
        "Debate_ID": row.get("Debate ID"),
        "Name": row.get("Name"),
        "Type": row.get("Type"),
        "Party": row.get("Party"),
        "Origin": row.get("Origin"),
        "Country": row.get("Country"),
        "Position": row.get("Position"),
        "Quote": speech,
        "postinvasion": row.get("postinvasion"),
        "Pos_Reg": row.get("Pos.Reg"),
        "Region": row.get("Region"),

        "geopolitical": parsed.get("geopolitical", False),
        "economic": parsed.get("economic", False),
        "normative": parsed.get("normative", False),
        "institutional": parsed.get("institutional", False),
        "procedural": parsed.get("procedural", False),
        "none": parsed.get("none", False)
    })

100%|███████████████████████████████████████| 1707/1707 [28:03<00:00,  1.01it/s]


In [53]:
# Tallennus
out_df = pd.DataFrame(results)

out_df.to_csv(OUTPUT_CSV, index=False)